# mms_sensitivity.ipynb

PINN matrix-flux reconstruction sensitivity study for the conforming single-fracture MMS benchmark.

Scope: one conforming mesh (`h = 1/64`), one cached CG-LMDFM solve, matrix-flux reconstruction only, no transport solve, no mesh-convergence loop. Experiments here are depth-width and activation sensitivity only.

The notebook is intentionally split into compute and render cells. Compute cells write CSV/NPZ results to disk. Render cells load those files and do not depend on trained model objects or compute-cell variables.


In [ ]:
# Cell 0 - Setup
from __future__ import annotations

import csv
import itertools
import json
import math
import platform
import time
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mpi4py import MPI
import dolfinx
from dolfinx import fem, mesh, geometry
from dolfinx.fem import petsc
import ufl
import basix
try:
    from dolfinx.io import gmsh as gmshio
except ImportError:
    from dolfinx.io import gmshio

import torch
import torch.nn as nn
import torch.nn.functional as F

# Fixed conforming MMS case.
REF = 6                         # regular_mesh_with_fracture_6.msh, h = 1/64
FEM_ORDER = 2                   # same as the split-diagnostic MMS notebook
MESH_FILE = f'regular_mesh_with_fracture_{REF}.msh'

# Paths. Works when launched from this notebook's folder or from the project root.
NOTEBOOK_DIR = Path.cwd()
_candidate_dir = Path.cwd() / 'fenicsx' / 'code' / 'fracture problem'
if not (NOTEBOOK_DIR / MESH_FILE).exists() and (_candidate_dir / MESH_FILE).exists():
    NOTEBOOK_DIR = _candidate_dir
MESH_PATH = NOTEBOOK_DIR / MESH_FILE
RESULTS_DIR = NOTEBOOK_DIR / 'results'
CURVE_DIR = RESULTS_DIR / 'curves'
CACHE_DIR = RESULTS_DIR / 'cache'
for _p in (RESULTS_DIR, CURVE_DIR, CACHE_DIR):
    _p.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / 'mms_sensitivity.csv'
CG_CACHE = CACHE_DIR / 'mms_cg_lmdfm_ref6_order2_cache.npz'
FRAC_A = np.array([0.0, 0.0], dtype=float)
FRAC_B = np.array([1.0, 1.0], dtype=float)
ALPHA = 1.0
K_M_VALUE = 1.0
K_F_VALUE = 100.0
TIP_FRAC = 0.10
N_REAL = 256
N_PTS_PER_EDGE = 3
MASK_FACTOR_START = 0.15
MASK_FACTOR_END = 1.15

# Fixed training protocol.
SEEDS = [7, 13, 23]
MAX_ITERS = 5000                # fixed-budget comparison
LOSS_TOL = 1.0e-3               # logged only; does not stop training
PRINT_EVERY = 250
CURVE_EVERY = 50
BATCH_DATA = 8192
BATCH_DIV = 4096
LR_PLUS = 2.0e-3
LR_MINUS = 2.0e-3
SCHED_GAMMA = 0.35
W_DATA = 5.0
W_DIV = 1.0
W_CONS_INT = 1.0
W_CONS_TIP = 1.0
CONS_INT_BETA = 1.0
CONS_R_FLOOR = 1.0e-5
CONS_INT_GAMMA = 4.0
W_JUMP = 1.0
W_SLOPE = 1.0e-4
CONS_RAMP_START = int(0.60 * MAX_ITERS)
CONS_RAMP_END = int(0.90 * MAX_ITERS)
BEST_START_ITER = CONS_RAMP_END

# Experiment 2 size is deliberately hard-coded after inspecting Experiment 1.
# Edit these two numbers after the Exp 1 render cells identify the knee.
EXP2_DEPTH = 3
EXP2_WIDTH = 32

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.set_default_dtype(torch.float32)
print('device:', DEVICE)

# Geometry helpers.
tau_np = (FRAC_B - FRAC_A) / np.linalg.norm(FRAC_B - FRAC_A)
normal_np = np.array([-tau_np[1], tau_np[0]], dtype=float)
L_GAMMA = float(np.linalg.norm(FRAC_B - FRAC_A))


def q_exact_xy(x0, y0):
    return np.sin(np.pi * x0) * np.sin(np.pi * y0)


def r_exact_xy(x0, y0):
    return (x0 - y0) / np.sqrt(2.0)


def grad_q_exact_xy(x0, y0):
    return np.column_stack([
        np.pi * np.cos(np.pi * x0) * np.sin(np.pi * y0),
        np.pi * np.sin(np.pi * x0) * np.cos(np.pi * y0),
    ])


def p_m_exact_xy(x0, y0):
    q = q_exact_xy(x0, y0)
    return q + ALPHA * np.abs(r_exact_xy(x0, y0)) * q


def exact_flux(points):
    pts = np.asarray(points, dtype=float).reshape(-1, 2)
    x0, y0 = pts[:, 0], pts[:, 1]
    q = q_exact_xy(x0, y0)
    grad_q = grad_q_exact_xy(x0, y0)
    r = r_exact_xy(x0, y0)
    sign_r = np.where(r >= 0.0, 1.0, -1.0)
    grad_r = np.array([1.0, -1.0], dtype=float) / np.sqrt(2.0)
    grad_p = grad_q + ALPHA * (sign_r[:, None] * grad_r[None, :] * q[:, None] + np.abs(r)[:, None] * grad_q)
    return -K_M_VALUE * grad_p


def lambda_exact_t(t):
    return -2.0 * ALPHA * np.sin(np.pi * np.asarray(t, dtype=float)) ** 2


def lambda_exact_s(s):
    return lambda_exact_t(np.asarray(s, dtype=float) / np.sqrt(2.0))


def f_m_exact_xy(x0, y0):
    q = q_exact_xy(x0, y0)
    r = r_exact_xy(x0, y0)
    sign_r = np.where(r >= 0.0, 1.0, -1.0)
    return (
        2.0 * np.pi**2 * q
        + 2.0 * ALPHA * np.pi**2 * np.abs(r) * q
        - ALPHA * np.sqrt(2.0) * np.pi * sign_r * np.sin(np.pi * (y0 - x0))
    )


def f_m_exact_points(points):
    pts = np.asarray(points, dtype=float).reshape(-1, 2)
    return f_m_exact_xy(pts[:, 0], pts[:, 1])


def f_gamma_exact_x(x):
    x = np.asarray(x, dtype=float)
    return -K_F_VALUE * np.pi**2 * np.cos(2.0 * np.pi * x) + lambda_exact_t(x)


def signed_distance_np(points):
    return (np.asarray(points, dtype=float)[:, :2] - FRAC_A[None, :]) @ normal_np


def s_coord_np(points):
    return (np.asarray(points, dtype=float)[:, :2] - FRAC_A[None, :]) @ tau_np


def real_points_np(s):
    s = np.asarray(s, dtype=float).reshape(-1)
    return FRAC_A[None, :] + s[:, None] * tau_np[None, :]


def polygon_signed_area(poly):
    poly = np.asarray(poly, dtype=float)
    x = poly[:, 0]
    y = poly[:, 1]
    return 0.5 * np.sum(x * np.roll(y, -1) - np.roll(x, -1) * y)


def order_polygon_vertices(pts):
    pts = np.asarray(pts, dtype=float)
    ctr = pts.mean(axis=0)
    ang = np.arctan2(pts[:, 1] - ctr[1], pts[:, 0] - ctr[0])
    poly = pts[np.argsort(ang)]
    if polygon_signed_area(poly) < 0.0:
        poly = poly[::-1]
    return poly


def edge_key(x0, x1, decimals=12):
    a = tuple(np.round(np.asarray(x0, dtype=float)[:2], decimals))
    b = tuple(np.round(np.asarray(x1, dtype=float)[:2], decimals))
    return tuple(sorted((a, b)))


def point_in_convex_polygon(pt, poly, tol=1.0e-12):
    pt = np.asarray(pt, dtype=float)[:2]
    poly = order_polygon_vertices(np.asarray(poly, dtype=float)[:, :2])
    for j in range(len(poly)):
        a = poly[j]
        b = poly[(j + 1) % len(poly)]
        edge = b - a
        cross = edge[0] * (pt[1] - a[1]) - edge[1] * (pt[0] - a[0])
        if cross < -tol:
            return False
    return True


def segment_edge_s_intersection(edge_a, edge_b, tol=1.0e-12):
    edge_a = np.asarray(edge_a, dtype=float)[:2]
    edge_b = np.asarray(edge_b, dtype=float)[:2]
    v = edge_b - edge_a
    A = np.column_stack((tau_np, -v))
    det = np.linalg.det(A)
    if abs(det) < tol:
        return None
    rhs = edge_a - FRAC_A
    s, u = np.linalg.solve(A, rhs)
    if -tol <= s <= L_GAMMA + tol and -tol <= u <= 1.0 + tol:
        return float(np.clip(s, 0.0, L_GAMMA))
    return None


def fracture_intervals_in_polygon(poly, tol=1.0e-11):
    poly = order_polygon_vertices(np.asarray(poly, dtype=float)[:, :2])
    candidates = [0.0, L_GAMMA]
    for j in range(len(poly)):
        s_int = segment_edge_s_intersection(poly[j], poly[(j + 1) % len(poly)])
        if s_int is not None:
            candidates.append(s_int)
    candidates = np.array(sorted(candidates), dtype=float)
    unique = []
    for s in candidates:
        if not unique or abs(s - unique[-1]) > tol:
            unique.append(float(s))
    intervals = []
    for s0, s1 in zip(unique[:-1], unique[1:]):
        if s1 - s0 <= tol:
            continue
        sm = 0.5 * (s0 + s1)
        if point_in_convex_polygon(real_points_np([sm])[0], poly, tol=1.0e-10):
            intervals.append((float(s0), float(s1)))
    return intervals


def outward_normal_len(x0, x1):
    edge = np.asarray(x1, dtype=float) - np.asarray(x0, dtype=float)
    return np.array([edge[1], -edge[0]], dtype=float)


def normal_from_segment_to_vertex(centroid, midpoint, vi):
    t = np.asarray(midpoint, dtype=float) - np.asarray(centroid, dtype=float)
    n1 = np.array([t[1], -t[0]], dtype=float)
    n2 = -n1
    seg_mid = 0.5 * (np.asarray(centroid, dtype=float) + np.asarray(midpoint, dtype=float))
    chosen = n1 if np.dot(n1, np.asarray(vi, dtype=float) - seg_mid) < 0.0 else n2
    return chosen / (np.linalg.norm(chosen) + 1.0e-30)


def integrate_source_polygon(poly):
    tri_bary = np.array([
        [1/3, 1/3, 1/3],
        [0.059715871789770, 0.470142064105115, 0.470142064105115],
        [0.470142064105115, 0.059715871789770, 0.470142064105115],
        [0.470142064105115, 0.470142064105115, 0.059715871789770],
        [0.797426985353087, 0.101286507323456, 0.101286507323456],
        [0.101286507323456, 0.797426985353087, 0.101286507323456],
        [0.101286507323456, 0.101286507323456, 0.797426985353087],
    ], dtype=float)
    tri_weights = np.array([
        0.225,
        0.132394152788506,
        0.132394152788506,
        0.132394152788506,
        0.125939180544827,
        0.125939180544827,
        0.125939180544827,
    ], dtype=float)
    poly = order_polygon_vertices(np.asarray(poly, dtype=float)[:, :2])
    total = 0.0
    p0 = poly[0]
    for j in range(1, len(poly) - 1):
        tri = np.vstack([p0, poly[j], poly[j + 1]])
        area = abs(polygon_signed_area(tri))
        qpts = tri_bary @ tri
        total += area * float(np.dot(tri_weights, f_m_exact_points(qpts)))
    return float(total)


def integrate_lambda_exact_interval(s0, s1, nq=4):
    if s1 - s0 <= 0.0:
        return 0.0
    xi, wi = np.polynomial.legendre.leggauss(int(nq))
    sq = 0.5 * (s0 + s1) + 0.5 * (s1 - s0) * xi
    return float(0.5 * (s1 - s0) * np.dot(wi, lambda_exact_s(sq)))


def residual_stats(values):
    vals = np.asarray(values, dtype=float).reshape(-1)
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return {'med': np.nan, 'max': np.nan}
    vals = np.abs(vals)
    return {'med': float(np.median(vals)), 'max': float(np.max(vals))}


def count_parameters(model):
    return int(sum(p.numel() for p in model.parameters() if p.requires_grad))


def load_results():
    if not RESULTS_CSV.exists():
        return pd.DataFrame()
    return pd.read_csv(RESULTS_CSV)


def write_experiment_results(experiment, rows):
    rows = list(rows)
    old = load_results()
    new = pd.DataFrame(rows)
    if old.empty:
        out = new
    else:
        old = old[old['experiment'] != experiment]
        out = pd.concat([old, new], ignore_index=True)
    out.to_csv(RESULTS_CSV, index=False)
    return out


def summarize_groups(df, group_cols, metric_cols):
    rows = []
    for keys, g in df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        row = {col: key for col, key in zip(group_cols, keys)}
        row['n_seeds'] = int(g['seed'].nunique())
        row['n_params'] = int(np.median(g['n_params']))
        for metric in metric_cols:
            row[f'{metric}_med'] = float(np.median(g[metric]))
            row[f'{metric}_min'] = float(np.min(g[metric]))
            row[f'{metric}_max'] = float(np.max(g[metric]))
        rows.append(row)
    return pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True)


## Cell 1 - CG-LMDFM solve, cache, and loader

Run this cell when the CG cache should be recomputed. The PINN experiment cells load `CG_CACHE` and do not re-solve the finite-element problem.


In [ ]:
# Cell 1 - CG solve (compute -> save) plus loader.

def _as_3d(points):
    pts = np.asarray(points, dtype=float)
    if pts.ndim == 1:
        pts = pts.reshape(1, -1)
    if pts.shape[1] == 3:
        return pts.copy()
    out = np.zeros((pts.shape[0], 3), dtype=float)
    out[:, :2] = pts[:, :2]
    return out


def _eval_fem_function(fun, points, domain=None):
    if domain is None:
        domain = fun.function_space.mesh
    pts3 = _as_3d(points)
    tree = geometry.bb_tree(domain, domain.topology.dim)
    candidates = geometry.compute_collisions_points(tree, pts3)
    colliding = geometry.compute_colliding_cells(domain, candidates, pts3)
    valid_ids, valid_points, valid_cells = [], [], []
    for i in range(pts3.shape[0]):
        links = colliding.links(i)
        if len(links) > 0:
            valid_ids.append(i)
            valid_points.append(pts3[i])
            valid_cells.append(links[0])
    if not valid_ids:
        return np.full((pts3.shape[0], 1), np.nan)
    valid_points = np.asarray(valid_points, dtype=float)
    valid_cells = np.asarray(valid_cells, dtype=np.int32)
    vals = np.asarray(fun.eval(valid_points, valid_cells))
    if vals.ndim == 1:
        vals = vals.reshape(len(valid_points), -1)
    out = np.full((pts3.shape[0], vals.shape[1]), np.nan)
    out[np.asarray(valid_ids, dtype=int), :] = vals
    return out


def _build_zeta_geometry(xmin, xmax, ymin, ymax, h_est):
    angle_deg = 45.0
    hu = 1.0 * h_est
    hv = 1.0 * h_est
    shift_fraction = np.asarray((0.31, 0.47), dtype=float)
    shift_u = float(shift_fraction[0] * hu)
    shift_v = float(shift_fraction[1] * hv)
    min_area = 1.0e-14

    def clip_axis(poly, axis, bound, keep_less, tol=1.0e-12):
        poly = [np.asarray(p, dtype=float) for p in poly]
        if not poly:
            return []
        def inside(p):
            return p[axis] <= bound + tol if keep_less else p[axis] >= bound - tol
        out = []
        prev = poly[-1]
        prev_in = inside(prev)
        for cur in poly:
            cur_in = inside(cur)
            if cur_in != prev_in:
                denom = cur[axis] - prev[axis]
                if abs(denom) > 1.0e-15:
                    t = (bound - prev[axis]) / denom
                    out.append(prev + t * (cur - prev))
            if cur_in:
                out.append(cur)
            prev, prev_in = cur, cur_in
        return out

    def clip_bbox(poly):
        out = [np.asarray(p, dtype=float) for p in poly]
        for axis, bound, keep_less in [(0, xmin, False), (0, xmax, True), (1, ymin, False), (1, ymax, True)]:
            out = clip_axis(out, axis, bound, keep_less)
            if len(out) == 0:
                return np.zeros((0, 2), dtype=float)
        return order_polygon_vertices(np.asarray(out, dtype=float))

    def grid_lines(lo, hi, H, shift, tol=1.0e-12):
        shift = float(np.mod(shift, H))
        lines = [float(lo), float(hi)]
        x = float(lo) + shift
        if x <= lo + tol:
            x += H
        while x < hi - tol:
            lines.append(float(x))
            x += H
        return np.asarray(sorted(lines), dtype=float)

    theta = np.deg2rad(angle_deg)
    e_u = np.array([np.cos(theta), np.sin(theta)], dtype=float)
    e_v = np.array([-np.sin(theta), np.cos(theta)], dtype=float)
    origin = np.array([xmin, ymin], dtype=float)
    bbox_poly = np.array([[xmin, ymin], [xmax, ymin], [xmax, ymax], [xmin, ymax]], dtype=float)
    bbox_uv = np.column_stack(((bbox_poly - origin) @ e_u, (bbox_poly - origin) @ e_v))
    u_lines = grid_lines(bbox_uv[:, 0].min(), bbox_uv[:, 0].max(), hu, shift_u)
    v_lines = grid_lines(bbox_uv[:, 1].min(), bbox_uv[:, 1].max(), hv, shift_v)
    polys, centers, areas = [], [], []
    for u0, u1 in zip(u_lines[:-1], u_lines[1:]):
        for v0, v1 in zip(v_lines[:-1], v_lines[1:]):
            corners_uv = np.array([[u0, v0], [u1, v0], [u1, v1], [u0, v1]], dtype=float)
            rect_xy = origin + corners_uv[:, 0:1] * e_u[None, :] + corners_uv[:, 1:2] * e_v[None, :]
            poly = clip_bbox(rect_xy)
            if len(poly) < 3:
                continue
            area = abs(polygon_signed_area(poly))
            if area <= min_area:
                continue
            polys.append(poly)
            centers.append(poly.mean(axis=0))
            areas.append(float(area))
    return polys, np.asarray(centers, dtype=float), np.asarray(areas, dtype=float), {
        'angle_deg': angle_deg,
        'hu': hu,
        'hv': hv,
        'shift_u': shift_u,
        'shift_v': shift_v,
    }


def _build_polygon_cv_quadrature(polys, prefix, edge_nq=4, lambda_integral_func=integrate_lambda_exact_interval):
    edge_xi, edge_wi = np.polynomial.legendre.leggauss(int(edge_nq))
    edge_xi = 0.5 * (edge_xi + 1.0)
    edge_wi = 0.5 * edge_wi
    points, nw, ids = [], [], []
    source = np.zeros(len(polys), dtype=float)
    lambda_src = np.zeros(len(polys), dtype=float)
    lambda_abs = np.zeros(len(polys), dtype=float)
    for pid, poly in enumerate(polys):
        poly = order_polygon_vertices(poly)
        source[pid] = integrate_source_polygon(poly)
        for s0, s1 in fracture_intervals_in_polygon(poly):
            val = lambda_integral_func(s0, s1, nq=4)
            lambda_src[pid] += val
            lambda_abs[pid] += abs(val)
        for j in range(len(poly)):
            x0 = poly[j]
            x1 = poly[(j + 1) % len(poly)]
            edge = x1 - x0
            n_len = outward_normal_len(x0, x1)
            for q, w in zip(edge_xi, edge_wi):
                points.append(x0 + q * edge)
                nw.append(w * n_len)
                ids.append(pid)
    return {
        f'{prefix}_points': np.asarray(points, dtype=float),
        f'{prefix}_nw': np.asarray(nw, dtype=float),
        f'{prefix}_ids': np.asarray(ids, dtype=np.int32),
        f'{prefix}_source': source,
        f'{prefix}_lambda': lambda_src,
        f'{prefix}_lambda_abs': lambda_abs,
    }


def compute_and_save_cg_cache():
    if MPI.COMM_WORLD.size != 1:
        raise RuntimeError('This sensitivity notebook expects one MPI rank.')

    msh, cell_markers, facet_markers = gmshio.read_from_msh(str(MESH_PATH), MPI.COMM_WORLD, 0, gdim=2)[0:3]
    omega = msh
    tdim = omega.topology.dim
    fdim = tdim - 1
    gamma_entities = facet_markers.find(2)
    gamma, gamma_to_omega, gamma_vertex_to_omega = mesh.create_submesh(omega, fdim, gamma_entities)[0:3]
    Gamma_tag = 2
    order = FEM_ORDER
    V_m = fem.functionspace(omega, ('Lagrange', order))
    V_f = fem.functionspace(gamma, ('Lagrange', order))
    V_l = fem.functionspace(gamma, ('Lagrange', order))
    W = ufl.MixedFunctionSpace(V_m, V_f, V_l)
    phi, psi, mu = ufl.TestFunctions(W)
    dp_m, dp_f, dl = ufl.TrialFunctions(W)
    p_m = fem.Function(V_m, name='p_m')
    p_f = fem.Function(V_f, name='p_f')
    lmbd = fem.Function(V_l, name='lmbd')

    x = ufl.SpatialCoordinate(omega)
    q_ufl = ufl.sin(np.pi * x[0]) * ufl.sin(np.pi * x[1])
    r_ufl = (x[0] - x[1]) / np.sqrt(2.0)
    abs_r_ufl = ufl.conditional(ufl.ge(r_ufl, 0.0), r_ufl, -r_ufl)
    sign_r_ufl = ufl.conditional(ufl.ge(r_ufl, 0.0), 1.0, -1.0)
    f_m_ufl = (
        2.0 * np.pi**2 * q_ufl
        + 2.0 * ALPHA * np.pi**2 * abs_r_ufl * q_ufl
        - ALPHA * np.sqrt(2.0) * np.pi * sign_r_ufl * ufl.sin(np.pi * (x[1] - x[0]))
    )

    def ff_callable(x):
        return f_gamma_exact_x(x[0])

    def k_callable(x):
        return (x[0] * 0.0 + K_M_VALUE)[np.newaxis, :]

    f_f = fem.Function(V_f)
    f_f.interpolate(ff_callable)
    k_m = fem.Function(V_m)
    k_m.interpolate(k_callable)
    k_f = fem.Function(V_f)
    k_f.x.array[:] = K_F_VALUE
    dx = ufl.Measure('dx', domain=omega)
    ds = ufl.Measure('ds', domain=omega, subdomain_data=facet_markers, subdomain_id=Gamma_tag)

    a_m0 = ufl.inner(k_m * ufl.grad(p_m), ufl.grad(phi)) * dx
    a_m1 = -lmbd * phi * ds
    L_m = f_m_ufl * phi * dx
    a_f0 = ufl.inner(k_f * ufl.grad(p_f), ufl.grad(psi)) * ds
    a_f1 = lmbd * psi * ds
    L_f = f_f * psi * ds
    a_l0 = p_m * mu * ds
    a_l1 = -p_f * mu * ds
    L_l = fem.Constant(omega, 0.0) * mu * ds
    F_form = (a_m0 + a_m1 - L_m) + (a_f0 + a_f1 - L_f) + (a_l0 + a_l1 - L_l)
    residual = ufl.extract_blocks(F_form)
    jac = ufl.derivative(F_form, p_m, dp_m) + ufl.derivative(F_form, p_f, dp_f) + ufl.derivative(F_form, lmbd, dl)
    J = ufl.extract_blocks(jac)

    coords = omega.geometry.x
    xmin, xmax = float(coords[:, 0].min()), float(coords[:, 0].max())
    ymin, ymax = float(coords[:, 1].min()), float(coords[:, 1].max())
    tol = 1.0e-10 * max(xmax - xmin, ymax - ymin)
    bnd_m = np.unique(np.concatenate([
        fem.locate_dofs_geometrical(V_m, lambda x: np.isclose(x[0], xmin, atol=tol)),
        fem.locate_dofs_geometrical(V_m, lambda x: np.isclose(x[0], xmax, atol=tol)),
        fem.locate_dofs_geometrical(V_m, lambda x: np.isclose(x[1], ymin, atol=tol)),
        fem.locate_dofs_geometrical(V_m, lambda x: np.isclose(x[1], ymax, atol=tol)),
    ]))
    p_m_bc = fem.Function(V_m)
    p_m_bc.x.array[:] = 0.0
    bc_pm = fem.dirichletbc(p_m_bc, bnd_m)
    tol_g = 1.0e-10 * np.max(np.ptp(gamma.geometry.x, axis=0))
    tip1 = fem.locate_dofs_geometrical(V_f, lambda x: np.logical_and(np.isclose(x[0], FRAC_A[0], atol=tol_g), np.isclose(x[1], FRAC_A[1], atol=tol_g)))
    tip2 = fem.locate_dofs_geometrical(V_f, lambda x: np.logical_and(np.isclose(x[0], FRAC_B[0], atol=tol_g), np.isclose(x[1], FRAC_B[1], atol=tol_g)))
    both_tips = np.unique(np.concatenate([tip1, tip2]))
    p_f_bc = fem.Function(V_f)
    p_f_bc.x.array[:] = 0.0
    bc_pf = fem.dirichletbc(p_f_bc, both_tips)

    nlp = petsc.NonlinearProblem(
        residual,
        u=[p_m, p_f, lmbd],
        J=J,
        bcs=[bc_pm, bc_pf],
        entity_maps=[gamma_to_omega],
        petsc_options={
            'snes_max_it': 200,
            'ksp_type': 'preonly',
            'pc_type': 'lu',
            'pc_factor_mat_solver_type': 'mumps',
            'mat_mumps_icntl_14': 120,
            'ksp_error_if_not_converged': True,
            'snes_error_if_not_converged': True,
        },
        petsc_options_prefix=f'mms_sensitivity_ref{REF}_',
    )
    t0 = time.perf_counter()
    nlp.solve()
    solve_walltime = time.perf_counter() - t0
    print('CG-LMDFM solved in', solve_walltime, 's; Newton iterations:', nlp.solver.getIterationNumber())

    omega.topology.create_connectivity(tdim, 0)
    omega.topology.create_connectivity(tdim, fdim)
    omega.topology.create_connectivity(fdim, tdim)
    omega.topology.create_connectivity(fdim, 0)
    c2v = omega.topology.connectivity(tdim, 0)
    cell_to_facet = omega.topology.connectivity(tdim, fdim)
    facet_to_cell = omega.topology.connectivity(fdim, tdim)
    facet_to_vertex = omega.topology.connectivity(fdim, 0)
    num_cells = omega.topology.index_map(tdim).size_local
    local_cell_vertices = [np.asarray(c2v.links(c), dtype=np.int32) for c in range(num_cells)]
    omega_xy = omega.geometry.x[:, :2]
    cell_polys = [order_polygon_vertices(omega_xy[verts]) for verts in local_cell_vertices]
    cell_centroids = np.vstack([poly.mean(axis=0) for poly in cell_polys])
    cell_areas = np.asarray([abs(polygon_signed_area(poly)) for poly in cell_polys])
    h_est = float(np.sqrt(np.mean(cell_areas)))

    Wq = fem.functionspace(omega, ('DG', order, (omega.geometry.dim,)))
    q_cg_expr = fem.Expression(-k_m * ufl.grad(p_m), Wq.element.interpolation_points)
    q_cg_fun = fem.Function(Wq, name='q_cg')
    q_cg_fun.interpolate(q_cg_expr)

    def q_cg_numpy(points):
        return _eval_fem_function(q_cg_fun, points, omega)[:, :2]

    LAMBDA_SIGN = 1.0

    def lambda_h_raw_on_s(s):
        pts = real_points_np(np.asarray(s, dtype=float).reshape(-1))
        return _eval_fem_function(lmbd, pts, gamma).reshape(-1)

    def lambda_h_on_s(s):
        return LAMBDA_SIGN * lambda_h_raw_on_s(s)

    def integrate_lambda_h_interval_raw(s0, s1, nq=4):
        if s1 - s0 <= 0.0:
            return 0.0
        xi, wi = np.polynomial.legendre.leggauss(int(nq))
        sq = 0.5 * (s0 + s1) + 0.5 * (s1 - s0) * xi
        return float(0.5 * (s1 - s0) * np.dot(wi, lambda_h_raw_on_s(sq)))

    def integrate_lambda_h_interval(s0, s1, nq=4):
        return LAMBDA_SIGN * integrate_lambda_h_interval_raw(s0, s1, nq=nq)

    # Gamma edge set and per-cell half multiplier source.
    gamma_edge_keys = set()
    lambda_h_cell_int_raw = np.zeros(num_cells, dtype=float)
    gamma_facet_cell_ids, gamma_facet_s0, gamma_facet_s1, gamma_facet_half_len = [], [], [], []
    lambda_nq = max(4, FEM_ORDER + 2)
    for gf in gamma_entities:
        verts = np.asarray(facet_to_vertex.links(int(gf)), dtype=np.int32)
        x0 = omega_xy[verts[0]]
        x1 = omega_xy[verts[1]]
        gamma_edge_keys.add(edge_key(x0, x1))
        length = float(np.linalg.norm(x1 - x0))
        s0, s1 = sorted([float(s_coord_np(x0[None, :])[0]), float(s_coord_np(x1[None, :])[0])])
        I_e_raw = integrate_lambda_h_interval_raw(s0, s1, nq=lambda_nq)
        for cell in facet_to_cell.links(int(gf)):
            if cell < num_cells:
                lambda_h_cell_int_raw[int(cell)] += 0.5 * I_e_raw
                gamma_facet_cell_ids.append(int(cell))
                gamma_facet_s0.append(s0)
                gamma_facet_s1.append(s1)
                gamma_facet_half_len.append(0.5 * length)

    # Cell source and edge quadrature for R_tau.
    edge_xi = np.array([0.5 * (1.0 - 1.0 / np.sqrt(3.0)), 0.5 * (1.0 + 1.0 / np.sqrt(3.0))], dtype=float)
    edge_wi = np.array([0.5, 0.5], dtype=float)
    tau_points, tau_nw, tau_cells, tau_is_gamma = [], [], [], []
    cell_source = np.zeros(num_cells, dtype=float)
    for cid, poly in enumerate(cell_polys):
        cell_source[cid] = integrate_source_polygon(poly)
        for j in range(len(poly)):
            x0 = poly[j]
            x1 = poly[(j + 1) % len(poly)]
            n_len = outward_normal_len(x0, x1)
            on_gamma = edge_key(x0, x1) in gamma_edge_keys
            for q, w in zip(edge_xi, edge_wi):
                tau_points.append(x0 + q * (x1 - x0))
                tau_nw.append(w * n_len)
                tau_cells.append(cid)
                tau_is_gamma.append(on_gamma)
    tau_points = np.asarray(tau_points, dtype=float)
    tau_nw = np.asarray(tau_nw, dtype=float)
    tau_cells = np.asarray(tau_cells, dtype=np.int32)
    tau_is_gamma = np.asarray(tau_is_gamma, dtype=bool)
    q_tau_cg = q_cg_numpy(tau_points)

    tau_active_for_sign = ~tau_is_gamma
    cell_flux_cg_non_gamma = np.zeros(num_cells, dtype=float)
    np.add.at(
        cell_flux_cg_non_gamma,
        tau_cells[tau_active_for_sign].astype(np.int64),
        np.sum(q_tau_cg[tau_active_for_sign] * tau_nw[tau_active_for_sign], axis=1),
    )
    R_plus = cell_flux_cg_non_gamma - cell_source - lambda_h_cell_int_raw
    R_minus = cell_flux_cg_non_gamma - cell_source + lambda_h_cell_int_raw
    if np.mean(np.abs(R_plus)) <= np.mean(np.abs(R_minus)):
        LAMBDA_SIGN = 1.0
    else:
        LAMBDA_SIGN = -1.0
    lambda_h_cell_int = LAMBDA_SIGN * lambda_h_cell_int_raw
    print('lambda sign:', LAMBDA_SIGN)
    print('mean |R_CG| with selected sign:', np.mean(np.abs(cell_flux_cg_non_gamma - cell_source - lambda_h_cell_int)))
    print('max  |R_CG| with selected sign:', np.max(np.abs(cell_flux_cg_non_gamma - cell_source - lambda_h_cell_int)))

    # Training data pool: edge samples away from the fracture.
    edge_sample_pts = []
    num_facets = omega.topology.index_map(fdim).size_local
    for f in range(num_facets):
        verts = np.asarray(facet_to_vertex.links(f), dtype=np.int32)
        if len(verts) < 2:
            continue
        x0 = omega_xy[verts[0]]
        x1 = omega_xy[verts[1]]
        ts = np.linspace(0.0, 1.0, N_PTS_PER_EDGE + 2)[1:-1]
        for t in ts:
            edge_sample_pts.append(x0 + t * (x1 - x0))
    X_data = np.asarray(edge_sample_pts, dtype=float)
    Q_data = q_cg_numpy(X_data)
    valid = np.all(np.isfinite(Q_data), axis=1)
    X_data = X_data[valid]
    Q_data = Q_data[valid]
    phi_data = signed_distance_np(X_data)
    data_mask_start = np.abs(phi_data) > MASK_FACTOR_START * h_est
    data_mask_final = np.abs(phi_data) > MASK_FACTOR_END * h_est

    X_div = cell_centroids.copy()
    F_div = f_m_exact_points(X_div)
    phi_div = signed_distance_np(X_div)
    div_mask_start = np.abs(phi_div) > MASK_FACTOR_START * h_est
    div_mask_final = np.abs(phi_div) > MASK_FACTOR_END * h_est

    # Real-fracture jump points.
    s_real = np.linspace(0.0, L_GAMMA, N_REAL)
    x_real = real_points_np(s_real)
    lambda_h_real = lambda_h_on_s(s_real)
    lambda_exact_real = lambda_exact_s(s_real)
    d_tip = np.minimum(s_real, L_GAMMA - s_real)
    w_tip = np.clip(d_tip / max(10.0 * h_est, 1.0e-14), 0.0, 1.0)

    # Bulk flux-error quadrature.
    metric_bary = np.array([
        [1/3, 1/3, 1/3],
        [0.059715871789770, 0.470142064105115, 0.470142064105115],
        [0.470142064105115, 0.059715871789770, 0.470142064105115],
        [0.470142064105115, 0.470142064105115, 0.059715871789770],
        [0.797426985353087, 0.101286507323456, 0.101286507323456],
        [0.101286507323456, 0.797426985353087, 0.101286507323456],
        [0.101286507323456, 0.101286507323456, 0.797426985353087],
    ], dtype=float)
    metric_w = np.array([0.225, 0.132394152788506, 0.132394152788506, 0.132394152788506,
                         0.125939180544827, 0.125939180544827, 0.125939180544827], dtype=float)
    metric_points, metric_weights = [], []
    for poly in cell_polys:
        # Mesh cells are triangles here.
        tri = np.asarray(poly[:3], dtype=float)
        area = abs(polygon_signed_area(tri))
        pts = metric_bary @ tri
        metric_points.append(pts)
        metric_weights.append(area * metric_w)
    metric_points = np.vstack(metric_points)
    metric_weights = np.concatenate(metric_weights)
    metric_q_exact = exact_flux(metric_points)

    # Deng-style vertex CV residual data for R_xi.
    vertex_key_to_id = {}
    vertices_xy = []
    cell_vids = []
    def get_vid(xy):
        key = tuple(np.round(np.asarray(xy, dtype=float)[:2], 12))
        if key not in vertex_key_to_id:
            vertex_key_to_id[key] = len(vertices_xy)
            vertices_xy.append(np.asarray(xy, dtype=float)[:2])
        return vertex_key_to_id[key]
    for poly in cell_polys:
        cell_vids.append(np.array([get_vid(xy) for xy in poly], dtype=np.int32))
    rxi_points, rxi_nw, rxi_vids = [], [], []
    rxi_source = np.zeros(len(vertices_xy), dtype=float)
    rxi_lambda = np.zeros(len(vertices_xy), dtype=float)
    for cid, poly_raw in enumerate(cell_polys):
        poly = order_polygon_vertices(poly_raw)
        centroid = poly.mean(axis=0)
        vids = cell_vids[cid]
        nloc = len(poly)
        for j, vid in enumerate(vids):
            vi = poly[j]
            mid_next = 0.5 * (vi + poly[(j + 1) % nloc])
            mid_prev = 0.5 * (vi + poly[(j - 1) % nloc])
            cv_poly = order_polygon_vertices(np.vstack([vi, mid_next, centroid, mid_prev]))
            rxi_source[vid] += integrate_source_polygon(cv_poly)
            for s0, s1 in fracture_intervals_in_polygon(cv_poly):
                rxi_lambda[vid] += integrate_lambda_h_interval(s0, s1, nq=4)
            for midp in (mid_next, mid_prev):
                seg = midp - centroid
                seg_len = float(np.linalg.norm(seg))
                if seg_len <= 1.0e-15:
                    continue
                n_out = normal_from_segment_to_vertex(centroid, midp, vi)
                for q, w in zip(edge_xi, edge_wi):
                    rxi_points.append(centroid + q * seg)
                    rxi_nw.append(w * seg_len * n_out)
                    rxi_vids.append(int(vid))
    vertices_xy = np.asarray(vertices_xy, dtype=float)
    bnd_mask = (
        np.isclose(vertices_xy[:, 0], xmin, atol=1.0e-12)
        | np.isclose(vertices_xy[:, 0], xmax, atol=1.0e-12)
        | np.isclose(vertices_xy[:, 1], ymin, atol=1.0e-12)
        | np.isclose(vertices_xy[:, 1], ymax, atol=1.0e-12)
    )
    rxi_interior_ids = np.where(~bnd_mask)[0].astype(np.int32)
    rxi_points = np.asarray(rxi_points, dtype=float)
    rxi_nw = np.asarray(rxi_nw, dtype=float)
    rxi_vids = np.asarray(rxi_vids, dtype=np.int32)
    q_rxi_cg = q_cg_numpy(rxi_points)

    # Rotated zeta CV residual data for R_zeta.
    zeta_polys, zeta_centers, zeta_areas, zeta_meta = _build_zeta_geometry(xmin, xmax, ymin, ymax, h_est)
    zeta_quad = _build_polygon_cv_quadrature(zeta_polys, 'zeta', edge_nq=4, lambda_integral_func=integrate_lambda_h_interval)
    q_zeta_cg = q_cg_numpy(zeta_quad['zeta_points'])

    # Save ragged polygons separately as padded arrays for optional plotting/debugging.
    zeta_counts = np.asarray([len(poly) for poly in zeta_polys], dtype=np.int32)
    max_zeta_vertices = int(zeta_counts.max()) if len(zeta_counts) else 0
    zeta_padded = np.full((len(zeta_polys), max_zeta_vertices, 2), np.nan, dtype=float)
    for i, poly in enumerate(zeta_polys):
        zeta_padded[i, :len(poly), :] = poly

    np.savez_compressed(
        CG_CACHE,
        ref=REF,
        fem_order=FEM_ORDER,
        h=1.0 / (2 ** REF),
        h_est=h_est,
        xmin=xmin, xmax=xmax, ymin=ymin, ymax=ymax,
        solve_walltime_s=solve_walltime,
        lambda_sign=LAMBDA_SIGN,
        cell_flux_cg_non_gamma=cell_flux_cg_non_gamma,
        cell_centroids=cell_centroids,
        cell_areas=cell_areas,
        cell_source=cell_source,
        lambda_h_cell_int=lambda_h_cell_int,
        frac_cell_ids=np.unique(np.asarray(gamma_facet_cell_ids, dtype=np.int32)),
        gamma_facet_cell_ids=np.asarray(gamma_facet_cell_ids, dtype=np.int32),
        gamma_facet_s0=np.asarray(gamma_facet_s0, dtype=float),
        gamma_facet_s1=np.asarray(gamma_facet_s1, dtype=float),
        gamma_facet_half_len=np.asarray(gamma_facet_half_len, dtype=float),
        tau_points=tau_points,
        tau_nw=tau_nw,
        tau_cells=tau_cells,
        tau_is_gamma=tau_is_gamma,
        q_tau_cg=q_tau_cg,
        X_data=X_data,
        Q_data=Q_data,
        data_mask_start=data_mask_start,
        data_mask_final=data_mask_final,
        X_div=X_div,
        F_div=F_div,
        div_mask_start=div_mask_start,
        div_mask_final=div_mask_final,
        s_real=s_real,
        x_real=x_real,
        lambda_h_real=lambda_h_real,
        lambda_exact_real=lambda_exact_real,
        w_tip=w_tip,
        metric_points=metric_points,
        metric_weights=metric_weights,
        metric_q_exact=metric_q_exact,
        rxi_points=rxi_points,
        rxi_nw=rxi_nw,
        rxi_vids=rxi_vids,
        rxi_source=rxi_source,
        rxi_lambda=rxi_lambda,
        rxi_interior_ids=rxi_interior_ids,
        q_rxi_cg=q_rxi_cg,
        zeta_points=zeta_quad['zeta_points'],
        zeta_nw=zeta_quad['zeta_nw'],
        zeta_ids=zeta_quad['zeta_ids'],
        zeta_source=zeta_quad['zeta_source'],
        zeta_lambda=zeta_quad['zeta_lambda'],
        zeta_lambda_abs=zeta_quad['zeta_lambda_abs'],
        zeta_centers=zeta_centers,
        zeta_areas=zeta_areas,
        zeta_polys=zeta_padded,
        zeta_poly_counts=zeta_counts,
        q_zeta_cg=q_zeta_cg,
        zeta_angle_deg=zeta_meta['angle_deg'],
        zeta_hu=zeta_meta['hu'],
        zeta_hv=zeta_meta['hv'],
        zeta_shift_u=zeta_meta['shift_u'],
        zeta_shift_v=zeta_meta['shift_v'],
    )
    print('saved CG cache:', CG_CACHE)
    print('data points:', len(X_data), 'div points:', len(X_div), 'fracture-adjacent cells:', len(np.unique(gamma_facet_cell_ids)))
    print('R_zeta CVs:', len(zeta_centers), 'R_xi vertices:', len(vertices_xy))


def load_cg_cache():
    if not CG_CACHE.exists():
        raise FileNotFoundError(f'Missing {CG_CACHE}. Run Cell 1 compute first.')
    raw = np.load(CG_CACHE, allow_pickle=False)
    return {k: raw[k] for k in raw.files}

# Recompute the CG cache when this cell is run.
compute_and_save_cg_cache()


## Cell 2 - PINN runner and save/load helpers

`run_reconstruction(config)` loads the cached CG arrays, trains one flux network on each side of the fracture, evaluates all metrics, writes the loss curve NPZ, and returns one tidy metrics row.


In [ ]:
# Cell 2 - Reconstruction runner.
class ActivationLayer(nn.Module):
    def __init__(self, name='tanh', adaptive_slope=True, n=5.0):
        super().__init__()
        self.name = str(name).lower()
        self.adaptive_slope = bool(adaptive_slope)
        if self.adaptive_slope:
            self.a = nn.Parameter(torch.tensor(1.0 / float(n), dtype=torch.float32))
        else:
            self.register_buffer('a', torch.tensor(1.0, dtype=torch.float32))

    def forward(self, x):
        z = self.a * x if self.adaptive_slope else x
        if self.name == 'tanh':
            return torch.tanh(z)
        if self.name == 'silu':
            return F.silu(z)
        if self.name == 'sin':
            return torch.sin(z)
        if self.name == 'relu':
            return F.relu(z)
        raise ValueError(f'Unknown activation: {self.name}')


class FluxNet(nn.Module):
    def __init__(self, in_dim=2, out_dim=2, width=32, depth=3, activation='tanh', adaptive_slope=True):
        super().__init__()
        layers = []
        last = in_dim
        for _ in range(int(depth)):
            layers.append(nn.Linear(last, int(width)))
            layers.append(ActivationLayer(activation, adaptive_slope=adaptive_slope))
            last = int(width)
        layers.append(nn.Linear(last, out_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

    def slope_recovery_term(self):
        slopes = [m.a for m in self.net if isinstance(m, ActivationLayer) and m.adaptive_slope]
        if not slopes:
            return torch.tensor(0.0, dtype=torch.float32, device=next(self.parameters()).device)
        return 1.0 / torch.mean(torch.exp(torch.stack(slopes)))


def _torch_cache(cache):
    def t(name, dtype=torch.float32):
        return torch.as_tensor(cache[name], dtype=dtype, device=DEVICE)
    out = {
        'X_data': t('X_data'),
        'Q_data': t('Q_data'),
        'data_idx_start': torch.as_tensor(np.nonzero(cache['data_mask_start'])[0], dtype=torch.long, device=DEVICE),
        'data_idx_final': torch.as_tensor(np.nonzero(cache['data_mask_final'])[0], dtype=torch.long, device=DEVICE),
        'X_div': t('X_div'),
        'F_div': t('F_div'),
        'div_idx_start': torch.as_tensor(np.nonzero(cache['div_mask_start'])[0], dtype=torch.long, device=DEVICE),
        'div_idx_final': torch.as_tensor(np.nonzero(cache['div_mask_final'])[0], dtype=torch.long, device=DEVICE),
        'x_real': t('x_real'),
        'lambda_exact_real': t('lambda_exact_real'),
        'lambda_h_real': t('lambda_h_real'),
        'w_tip': t('w_tip'),
        'tau_points': t('tau_points'),
        'tau_nw': t('tau_nw'),
        'tau_cells': torch.as_tensor(cache['tau_cells'], dtype=torch.long, device=DEVICE),
        'tau_is_gamma': torch.as_tensor(cache['tau_is_gamma'], dtype=torch.bool, device=DEVICE),
        'cell_source': t('cell_source'),
        'lambda_h_cell_int': t('lambda_h_cell_int'),
        'metric_points': t('metric_points'),
        'metric_weights': t('metric_weights'),
        'metric_q_exact': t('metric_q_exact'),
    }
    out['side_div'] = signed_distance_torch(out['X_div']) >= 0.0
    return out


xy_center_t = torch.tensor([0.5, 0.5], dtype=torch.float32, device=DEVICE)
xy_scale_t = torch.tensor(1.0, dtype=torch.float32, device=DEVICE)
frac_a_t = torch.tensor(FRAC_A, dtype=torch.float32, device=DEVICE)
normal_t = torch.tensor(normal_np, dtype=torch.float32, device=DEVICE)


def normalize_xy(x):
    return (x - xy_center_t) / xy_scale_t


def signed_distance_torch(x):
    return (x - frac_a_t) @ normal_t


def div_of_q(net_fn, x):
    x = x.clone().detach().requires_grad_(True)
    q = net_fn(x)
    dq0 = torch.autograd.grad(q[:, 0].sum(), x, create_graph=True)[0][:, 0]
    dq1 = torch.autograd.grad(q[:, 1].sum(), x, create_graph=True)[0][:, 1]
    return dq0 + dq1


def smoothstep01(t):
    t = float(np.clip(t, 0.0, 1.0))
    return t * t * (3.0 - 2.0 * t)


def config_ids(config):
    group_id = f"L{int(config['depth'])}_W{int(config['width'])}_{config['activation']}_adaptive{int(bool(config['adaptive_slope']))}"
    config_id = f"{group_id}_seed{int(config['seed'])}"
    return group_id, config_id


def compute_edge_residual_numpy(q_func, points, nw, ids, source, lambda_src, n_items):
    qv = q_func(points)
    edge_flux = np.sum(qv * nw, axis=1)
    flux = np.zeros(int(n_items), dtype=float)
    np.add.at(flux, ids.astype(np.int64), edge_flux)
    return flux - source - lambda_src


def run_reconstruction(config):
    cache = load_cg_cache()
    tc = _torch_cache(cache)
    seed = int(config['seed'])
    np.random.seed(seed)
    torch.manual_seed(seed)
    if DEVICE.type == 'cuda':
        torch.cuda.manual_seed_all(seed)

    q_plus_net = FluxNet(width=int(config['width']), depth=int(config['depth']), activation=config['activation'], adaptive_slope=bool(config['adaptive_slope'])).to(DEVICE)
    q_minus_net = FluxNet(width=int(config['width']), depth=int(config['depth']), activation=config['activation'], adaptive_slope=bool(config['adaptive_slope'])).to(DEVICE)

    def q_plus(x):
        return q_plus_net(normalize_xy(x))

    def q_minus(x):
        return q_minus_net(normalize_xy(x))

    def q_piecewise(x):
        phi = signed_distance_torch(x)
        qp = q_plus(x)
        qm = q_minus(x)
        return torch.where((phi >= 0.0).reshape(-1, 1), qp, qm)

    optimizer_plus = torch.optim.Adam(q_plus_net.parameters(), lr=LR_PLUS)
    optimizer_minus = torch.optim.Adam(q_minus_net.parameters(), lr=LR_MINUS)
    scheduler_plus = torch.optim.lr_scheduler.MultiStepLR(optimizer_plus, milestones=[int(0.45 * MAX_ITERS), int(0.75 * MAX_ITERS)], gamma=SCHED_GAMMA)
    scheduler_minus = torch.optim.lr_scheduler.MultiStepLR(optimizer_minus, milestones=[int(0.45 * MAX_ITERS), int(0.75 * MAX_ITERS)], gamma=SCHED_GAMMA)

    frac_cell_ids = cache['frac_cell_ids'].astype(np.int64)
    frac_local_of_cell = {int(cid): i for i, cid in enumerate(frac_cell_ids)}
    all_tau_cells = cache['tau_cells'].astype(np.int64)
    tau_cell_mask = np.isin(all_tau_cells, frac_cell_ids)
    tau_cell_mask_t = torch.as_tensor(tau_cell_mask, dtype=torch.bool, device=DEVICE)
    frac_tau_points = tc['tau_points'][tau_cell_mask_t]
    frac_tau_nw = tc['tau_nw'][tau_cell_mask_t]
    frac_tau_is_gamma = tc['tau_is_gamma'][tau_cell_mask_t]
    frac_tau_cells_np = np.array([frac_local_of_cell[int(cid)] for cid in all_tau_cells[tau_cell_mask]], dtype=np.int64)
    frac_tau_cells = torch.as_tensor(frac_tau_cells_np, dtype=torch.long, device=DEVICE)
    frac_source = tc['cell_source'][torch.as_tensor(frac_cell_ids, dtype=torch.long, device=DEVICE)]
    frac_lambda = tc['lambda_h_cell_int'][torch.as_tensor(frac_cell_ids, dtype=torch.long, device=DEVICE)]
    nf_mask = ~frac_tau_is_gamma
    frac_points_nf = frac_tau_points[nf_mask]
    frac_nw_nf = frac_tau_nw[nf_mask]
    frac_cells_nf = frac_tau_cells[nf_mask]
    q_cg_nf = torch.as_tensor(cache['q_tau_cg'][tau_cell_mask][~cache['tau_is_gamma'][tau_cell_mask]], dtype=torch.float32, device=DEVICE)
    flux_cg_q = torch.sum(q_cg_nf * frac_nw_nf, dim=1)
    flux_cg = torch.zeros(len(frac_cell_ids), device=DEVICE)
    flux_cg.scatter_add_(0, frac_cells_nf, flux_cg_q)
    frac_R_cg = flux_cg - frac_source - frac_lambda
    frac_s = s_coord_np(cache['cell_centroids'][frac_cell_ids])
    frac_tip_np = (frac_s < TIP_FRAC * L_GAMMA) | (frac_s > (1.0 - TIP_FRAC) * L_GAMMA)
    frac_tip = torch.as_tensor(frac_tip_np, dtype=torch.bool, device=DEVICE)
    frac_int = ~frac_tip

    def mse(x):
        return torch.mean(x ** 2)

    def curriculum(iteration):
        if CONS_RAMP_END <= CONS_RAMP_START:
            ramp = 1.0
        else:
            ramp = smoothstep01((iteration - CONS_RAMP_START) / (CONS_RAMP_END - CONS_RAMP_START))
        use_final_mask = iteration >= CONS_RAMP_START
        return ramp, (tc['data_idx_final'] if use_final_mask else tc['data_idx_start']), (tc['div_idx_final'] if use_final_mask else tc['div_idx_start'])

    curves = []
    history_last = {}
    best = {'loss': float('inf'), 'plus': None, 'minus': None, 'iteration': 0}
    iters_to_tol = MAX_ITERS
    hit_tol = False
    train_t0 = time.perf_counter()

    for iteration in range(1, MAX_ITERS + 1):
        ramp, data_pool, div_pool = curriculum(iteration)
        if data_pool.numel() == 0 or div_pool.numel() == 0:
            raise RuntimeError('A distance mask removed all data or div points.')

        optimizer_plus.zero_grad(set_to_none=True)
        optimizer_minus.zero_grad(set_to_none=True)

        idx_d = data_pool[torch.randint(0, data_pool.numel(), (min(BATCH_DATA, data_pool.numel()),), device=DEVICE)]
        loss_data = mse(q_piecewise(tc['X_data'][idx_d]) - tc['Q_data'][idx_d])

        idx_v = div_pool[torch.randint(0, div_pool.numel(), (min(BATCH_DIV, div_pool.numel()),), device=DEVICE)]
        x_div_b = tc['X_div'][idx_v]
        f_div_b = tc['F_div'][idx_v]
        side_b = tc['side_div'][idx_v]
        loss_div = torch.tensor(0.0, device=DEVICE)
        Xp = x_div_b[side_b]
        fp = f_div_b[side_b]
        Xm = x_div_b[~side_b]
        fm = f_div_b[~side_b]
        if Xp.numel() > 0:
            loss_div = loss_div + mse(div_of_q(q_plus, Xp) - fp)
        if Xm.numel() > 0:
            loss_div = loss_div + mse(div_of_q(q_minus, Xm) - fm)

        q_nf = q_piecewise(frac_points_nf)
        flux_q = torch.sum(q_nf * frac_nw_nf, dim=1)
        flux_K = torch.zeros(len(frac_cell_ids), device=DEVICE)
        flux_K.scatter_add_(0, frac_cells_nf, flux_q)
        R_rec_frac = flux_K - frac_source - frac_lambda
        loss_cons_frac = mse(R_rec_frac)
        loss_cons_int = torch.tensor(0.0, device=DEVICE)
        if frac_int.any().item():
            R_rec_int = R_rec_frac[frac_int]
            R_cg_int = frac_R_cg[frac_int]
            denom = torch.clamp(torch.abs(R_cg_int), min=CONS_R_FLOOR)
            r_norm = R_rec_int / denom
            excess = torch.relu(torch.abs(R_rec_int) - CONS_INT_BETA * torch.abs(R_cg_int))
            loss_cons_int = torch.mean(r_norm ** 2) + CONS_INT_GAMMA * torch.mean((excess / denom) ** 2)
        loss_cons_tip = torch.tensor(0.0, device=DEVICE)
        if frac_tip.any().item():
            loss_cons_tip = mse(R_rec_frac[frac_tip])

        qp_real = q_plus(tc['x_real'])
        qm_real = q_minus(tc['x_real'])
        jump_real = torch.sum((qp_real - qm_real) * normal_t.reshape(1, 2), dim=1)
        loss_jump = torch.mean(tc['w_tip'] * (jump_real - tc['lambda_exact_real']) ** 2)
        loss_slope = q_plus_net.slope_recovery_term() + q_minus_net.slope_recovery_term()
        loss = (
            W_DATA * loss_data
            + W_DIV * loss_div
            + ramp * W_CONS_INT * loss_cons_int
            + ramp * W_CONS_TIP * loss_cons_tip
            + W_JUMP * loss_jump
            + W_SLOPE * loss_slope
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(q_plus_net.parameters(), max_norm=1.0)
        torch.nn.utils.clip_grad_norm_(q_minus_net.parameters(), max_norm=1.0)
        optimizer_plus.step()
        optimizer_minus.step()
        scheduler_plus.step()
        scheduler_minus.step()

        loss_float = float(loss.detach().cpu())
        if (not hit_tol) and loss_float <= LOSS_TOL:
            iters_to_tol = iteration
            hit_tol = True
        if iteration >= BEST_START_ITER and loss_float < best['loss']:
            best['loss'] = loss_float
            best['iteration'] = iteration
            best['plus'] = {k: v.detach().cpu().clone() for k, v in q_plus_net.state_dict().items()}
            best['minus'] = {k: v.detach().cpu().clone() for k, v in q_minus_net.state_dict().items()}
        if iteration == 1 or iteration % CURVE_EVERY == 0 or iteration == MAX_ITERS:
            row = {
                'iter': iteration,
                'total': loss_float,
                'data': float(loss_data.detach().cpu()),
                'div': float(loss_div.detach().cpu()),
                'cons_frac': float(loss_cons_frac.detach().cpu()),
                'cons_int': float(loss_cons_int.detach().cpu()),
                'cons_tip': float(loss_cons_tip.detach().cpu()),
                'jump': float(loss_jump.detach().cpu()),
                'slope': float(loss_slope.detach().cpu()),
                'ramp': float(ramp),
            }
            curves.append(row)
            history_last = row
        if iteration == 1 or iteration % PRINT_EVERY == 0:
            print(f"{config_ids(config)[1]} iter={iteration:5d} total={loss_float:.3e} data={float(loss_data.detach().cpu()):.3e} div={float(loss_div.detach().cpu()):.3e} jump={float(loss_jump.detach().cpu()):.3e} cons={float(loss_cons_frac.detach().cpu()):.3e}")

    walltime_s = time.perf_counter() - train_t0
    if best['plus'] is not None:
        q_plus_net.load_state_dict(best['plus'])
        q_minus_net.load_state_dict(best['minus'])
    best_iteration = int(best['iteration'] or MAX_ITERS)

    @torch.no_grad()
    def q_numpy(points):
        pts = torch.as_tensor(np.asarray(points, dtype=np.float32), dtype=torch.float32, device=DEVICE)
        return q_piecewise(pts).detach().cpu().numpy()

    @torch.no_grad()
    def jump_numpy(points):
        pts = torch.as_tensor(np.asarray(points, dtype=np.float32), dtype=torch.float32, device=DEVICE)
        jp = torch.sum((q_plus(pts) - q_minus(pts)) * normal_t.reshape(1, 2), dim=1)
        return jp.detach().cpu().numpy()

    q_metric = q_numpy(cache['metric_points'])
    diff2 = np.sum((q_metric - cache['metric_q_exact']) ** 2, axis=1)
    exact2 = np.sum(cache['metric_q_exact'] ** 2, axis=1)
    flux_err = float(np.sqrt(np.dot(cache['metric_weights'], diff2) / max(np.dot(cache['metric_weights'], exact2), 1.0e-30)))

    jump_rec = jump_numpy(cache['x_real'])
    jump_err = float(np.sqrt(np.trapz((jump_rec - cache['lambda_exact_real']) ** 2, cache['s_real'])))
    jump_err_lh = float(np.sqrt(np.trapz((jump_rec - cache['lambda_h_real']) ** 2, cache['s_real'])))

    tau_active = ~cache['tau_is_gamma']
    Rtau = compute_edge_residual_numpy(
        q_numpy,
        cache['tau_points'][tau_active],
        cache['tau_nw'][tau_active],
        cache['tau_cells'][tau_active],
        cache['cell_source'],
        cache['lambda_h_cell_int'],
        len(cache['cell_source']),
    )
    Rtau_s = residual_stats(Rtau)

    Rxi = compute_edge_residual_numpy(
        q_numpy,
        cache['rxi_points'],
        cache['rxi_nw'],
        cache['rxi_vids'],
        cache['rxi_source'],
        cache['rxi_lambda'],
        len(cache['rxi_source']),
    )
    Rxi_vals = Rxi[cache['rxi_interior_ids'].astype(np.int64)]
    Rxi_s = residual_stats(Rxi_vals)

    Rzeta = compute_edge_residual_numpy(
        q_numpy,
        cache['zeta_points'],
        cache['zeta_nw'],
        cache['zeta_ids'],
        cache['zeta_source'],
        cache['zeta_lambda'],
        len(cache['zeta_source']),
    )
    Rzeta_s = residual_stats(Rzeta)

    group_id, config_id = config_ids(config)
    curve_path = CURVE_DIR / f'{config_id}.npz'
    curve_df = pd.DataFrame(curves)
    np.savez_compressed(curve_path, **{col: curve_df[col].to_numpy() for col in curve_df.columns})

    row = {
        'config_id': config_id,
        'group_id': group_id,
        'experiment': config.get('experiment', ''),
        'architecture': 'single',
        'depth': int(config['depth']),
        'width': int(config['width']),
        'activation': str(config['activation']),
        'adaptive_slope': bool(config['adaptive_slope']),
        'seed': seed,
        'flux_err': flux_err,
        'jump_err': jump_err,
        'jump_err_lh': jump_err_lh,
        'Rtau_med': Rtau_s['med'],
        'Rtau_max': Rtau_s['max'],
        'Rxi_med': Rxi_s['med'],
        'Rxi_max': Rxi_s['max'],
        'Rzeta_med': Rzeta_s['med'],
        'Rzeta_max': Rzeta_s['max'],
        'iters_to_tol': int(iters_to_tol),
        'hit_tol': bool(hit_tol),
        'walltime_s': float(walltime_s),
        'n_params': count_parameters(q_plus_net) + count_parameters(q_minus_net),
        'best_iter': best_iteration,
        'best_loss': float(best['loss']),
        'last_loss': float(history_last.get('total', np.nan)),
        'device': str(DEVICE),
        'torch_version': torch.__version__,
        'numpy_version': np.__version__,
        'dolfinx_version': getattr(dolfinx, '__version__', 'unknown'),
        'python_version': platform.python_version(),
        'curve_file': str(curve_path.relative_to(NOTEBOOK_DIR)),
    }
    print('metrics:', {k: row[k] for k in ['config_id', 'flux_err', 'jump_err', 'Rtau_med', 'Rzeta_med', 'walltime_s']})
    return row


## Experiment 1 - Depth-width compute

Fixed: `activation='tanh'`, `adaptive_slope=True`, default losses, uniform collocation, and the cached conforming MMS solve.


In [ ]:
# Cell 3a - Experiment 1 compute -> CSV/NPZ.
exp1_rows = []
for depth, width, seed in itertools.product([2, 3, 4], [16, 32, 64], SEEDS):
    cfg = {
        'experiment': 'exp1_depth_width',
        'depth': depth,
        'width': width,
        'activation': 'tanh',
        'adaptive_slope': True,
        'seed': seed,
    }
    exp1_rows.append(run_reconstruction(cfg))
write_experiment_results('exp1_depth_width', exp1_rows)
print('wrote', len(exp1_rows), 'rows to', RESULTS_CSV)


## Experiment 1 - Render table

Render-only: loads `results/mms_sensitivity.csv` and reports medians over seeds. Safe to run after a kernel restart once Cell 0 has run.


In [ ]:
# Cell 3b - Experiment 1 render table only.
df = load_results()
exp1 = df[df['experiment'] == 'exp1_depth_width'].copy()
if exp1.empty:
    raise FileNotFoundError('No Experiment 1 rows found. Run Cell 3a first.')
metric_cols = ['flux_err', 'jump_err', 'Rtau_med', 'Rtau_max', 'Rzeta_med', 'walltime_s']
exp1_table = summarize_groups(exp1, ['depth', 'width'], metric_cols)
show_cols = [
    'depth', 'width', 'n_seeds', 'n_params',
    'flux_err_med', 'flux_err_min', 'flux_err_max',
    'jump_err_med', 'Rtau_med_med', 'Rtau_max_med', 'Rzeta_med_med', 'walltime_s_med',
]
display(exp1_table[show_cols])


## Experiment 1 - Render knee figure

Render-only: log-log plot of flux error and `Rtau_med` versus parameter count.


In [ ]:
# Cell 3c - Experiment 1 render figure only.
df = load_results()
exp1 = df[df['experiment'] == 'exp1_depth_width'].copy()
if exp1.empty:
    raise FileNotFoundError('No Experiment 1 rows found. Run Cell 3a first.')
summary = summarize_groups(exp1, ['depth', 'width'], ['flux_err', 'Rtau_med'])
fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.2), constrained_layout=True)
for ax, metric, ylabel in [
    (axes[0], 'flux_err_med', r'relative flux error'),
    (axes[1], 'Rtau_med_med', r'median $|R_\tau|$'),
]:
    for depth, g in summary.groupby('depth'):
        g = g.sort_values('n_params')
        ax.loglog(g['n_params'], g[metric], 'o-', lw=1.8, ms=5, label=f'L={depth}')
        for _, row in g.iterrows():
            ax.annotate(f"W={int(row['width'])}", (row['n_params'], row[metric]), textcoords='offset points', xytext=(4, 4), fontsize=9)
    ax.set_xlabel('trainable parameters')
    ax.set_ylabel(ylabel)
    ax.grid(True, which='both', alpha=0.22)
    ax.legend(frameon=False)
fig.suptitle('Experiment 1: depth-width sensitivity')
plt.show()


## Experiment 2 - Activation compute

Edit `EXP2_DEPTH` and `EXP2_WIDTH` in Cell 0 after inspecting Experiment 1. Fixed-size comparison over activations and adaptive slope.


In [ ]:
# Cell 4a - Experiment 2 compute -> CSV/NPZ.
exp2_rows = []
for activation, adaptive_slope, seed in itertools.product(['tanh', 'silu', 'sin', 'relu'], [False, True], SEEDS):
    cfg = {
        'experiment': 'exp2_activation',
        'depth': EXP2_DEPTH,
        'width': EXP2_WIDTH,
        'activation': activation,
        'adaptive_slope': adaptive_slope,
        'seed': seed,
    }
    exp2_rows.append(run_reconstruction(cfg))
write_experiment_results('exp2_activation', exp2_rows)
print('wrote', len(exp2_rows), 'rows to', RESULTS_CSV)


## Experiment 2 - Render table

Render-only: medians over seeds for each activation and adaptive-slope setting.


In [ ]:
# Cell 4b - Experiment 2 render table only.
df = load_results()
exp2 = df[df['experiment'] == 'exp2_activation'].copy()
if exp2.empty:
    raise FileNotFoundError('No Experiment 2 rows found. Run Cell 4a first.')
metric_cols = ['flux_err', 'jump_err', 'Rtau_med', 'Rxi_med', 'Rzeta_med', 'walltime_s']
exp2_table = summarize_groups(exp2, ['activation', 'adaptive_slope'], metric_cols)
show_cols = [
    'activation', 'adaptive_slope', 'n_seeds', 'n_params',
    'flux_err_med', 'flux_err_min', 'flux_err_max',
    'jump_err_med', 'Rtau_med_med', 'Rxi_med_med', 'Rzeta_med_med', 'walltime_s_med',
]
display(exp2_table[show_cols])


## Experiment 2 - Render convergence curves

Render-only: loads the per-run NPZ curves and plots median total loss per activation/slope group.


In [ ]:
# Cell 4c - Experiment 2 render convergence curves only.
df = load_results()
exp2 = df[df['experiment'] == 'exp2_activation'].copy()
if exp2.empty:
    raise FileNotFoundError('No Experiment 2 rows found. Run Cell 4a first.')
fig, ax = plt.subplots(figsize=(8.6, 5.0), constrained_layout=True)
for (activation, adaptive), g in exp2.groupby(['activation', 'adaptive_slope']):
    curves = []
    iter_ref = None
    for _, row in g.iterrows():
        path = NOTEBOOK_DIR / row['curve_file']
        if not path.exists():
            continue
        arr = np.load(path)
        it = arr['iter']
        total = arr['total']
        if iter_ref is None:
            iter_ref = it
        if len(it) == len(iter_ref) and np.all(it == iter_ref):
            curves.append(total)
        else:
            curves.append(np.interp(iter_ref, it, total))
    if not curves:
        continue
    y = np.median(np.vstack(curves), axis=0)
    label = f'{activation}, adaptive={bool(adaptive)}'
    ax.semilogy(iter_ref, y, lw=1.9, label=label)
ax.axhline(LOSS_TOL, color='0.35', lw=0.9, ls='--', label='logged tolerance')
ax.set_xlabel('iteration')
ax.set_ylabel('median total loss')
ax.grid(True, which='both', alpha=0.22)
ax.legend(frameon=False, ncol=2)
ax.set_title(f'Experiment 2 convergence curves, L={EXP2_DEPTH}, W={EXP2_WIDTH}')
plt.show()
